## Initial Setup

### Libraries and Dependencies

In [ ]:
!pip -q install \
  langgraph==0.2.39 \
  langchain==0.3.14 \
  langchain-core==0.3.29 \
  langchain-community==0.3.14 \
  langsmith==0.1.147 \
  openai==1.59.6 \
  langchain-openai==0.2.14 \
  pydantic==2.10.4 \
  pandas==2.2.3 \
  numpy==2.2.1

### Imports

In [ ]:
import os              # environment variables / runtime config
import re              # text normalization + lightweight pattern matching
import json            # serialize/parse tool outputs + structured data interchange
import sqlite3         # connect/query the SQLite database
import pandas as pd    # load/query CSV reference data
import numpy as np     # numeric utilities
from typing import Any, Dict, List, Optional, TypedDict, Tuple  # type hints + structured state for agent workflow

from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage  # chat + tool observation messages
from langchain_core.prompts import ChatPromptTemplate  # consistent prompt templates
from langchain_core.tools import tool                 # define LLM-callable tools
from langchain_core.tools.base import BaseTool                     # base class for tools with metadata
from langchain_core.output_parsers import JsonOutputParser  # parse LLM outputs into JSON

from langchain_openai import ChatOpenAI  # OpenAI chat model wrapper

from langgraph.graph import StateGraph, END  # build the LangGraph state machine + termination node

from IPython.display import Markdown, Image, display  # visualize the graph / notebook-friendly display

In [ ]:
# Display Markdown formatted string in the notebook output.
# This function takes a string as input and uses IPython.display.Markdown
# to render it as Markdown, providing a more visually appealing output.
def printmd(string):
    display(Markdown(string))

In [ ]:
# Identify the environment to switch the file paths either to use Google Drive or Local
import sys
# Check if the operating system is Windows
if sys.platform.startswith('win'):
    printmd("Running on Windows")
    resourcePath=""
else:
    # Assume it's a Linux-based environment, likely Google Colab
    printmd("Running on Linux most likely Colab")
    # Mount Google Drive for Colab environments to access files
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    # Define the base path for resources in Google Drive
    resourcePath="/content/drive/MyDrive/ColabAssignments/ReActEHR/"

### Generic Utility functions

In [ ]:
def normalizeText(text: str) -> str:
    """Normalize text for consistent matching (lowercase + trimmed + whitespace-collapsed).
        Inputs:
        s (str): Any input text (will be cast to string).

    Output:
        str: Normalized text with:
            - leading/trailing whitespace removed
            - converted to lowercase
            - internal / multiple whitespace collapsed to single spaces
    """
    return re.sub(
        r"\s+", " ", str(text).strip().lower()
    )  # Convert to lowercase and remove special characters


def toJson(obj: Any) -> str:
    """Convert an object to a JSON string.
        Inputs:
        obj (Any): Any Python object (dict/list/str/etc.). If the object contains
            non-JSON-serializable values (e.g., datetime, numpy types), they are
            converted to strings via default=str.

    Output:
        str: JSON-formatted string (UTF-8 friendly, ensure_ascii=False).
    """
    return json.dumps(obj, ensure_ascii=False, default=str)

### CSV Utility functions

In [ ]:
def readCSV(filePath: str) -> pd.DataFrame:
    """Reads a CSV file and returns its content as a data frame.
    Inputs:
        filePath (str): The file path to the CSV file (e.g., "data/patients.csv").
    Output:
        pandas.DataFrame: A DataFrame containing the data from the CSV file.
    """
    print(f"Attempting to read CSV file: {filePath}")
    frame = pd.read_csv(filePath)  # Read the CSV file into a DataFrame
    # print(frame.head(1))  # Display the first few rows for verification
    print(
        f"Succesfully read {len(frame)} rows from {filePath}"
    )  # Log the number of rows read
    return frame

### Define and load the provided CSV Data

In [ ]:
TRUSTED_SOURCES_CSV = resourcePath + "datafiles/trusted_sources_catalog.csv"  # path to the trusted_sources_catalog.csv file
LAB_EXPLAIN_CSV = resourcePath + "datafiles/patient_friendly_lab_explanations.csv"  # path to the patient_friendly_lab_explanations.csv file
MED_EDU_CSV = (
    resourcePath + "datafiles/medication_education.csv"  # path to the medication_education.csv file
)
POLICY_CSV = (
    resourcePath + "datafiles/safety_policy_rules.csv"  # path to the safety_policy_rules.csv file
)

trustedSources = readCSV(TRUSTED_SOURCES_CSV)
labExplain = readCSV(LAB_EXPLAIN_CSV)
medEdu = readCSV(MED_EDU_CSV)
policyRules = readCSV(POLICY_CSV)

### Define and load the Database

In [ ]:
DB_PATH = resourcePath + "datafiles/health_portal.db"  # path to the SQLite database file
print(f"Connecting to database at: {DB_PATH}")
conn = sqlite3.connect(DB_PATH)  # Establish a connection to the database
cursor = conn.cursor()  # Create a cursor object to execute SQL queries
allTables = cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table';"
).fetchall()
for t in allTables:
    print(f"Table: {t[0]}")  # Log the name of each table in the database

### Database Utility functions

In [ ]:
def queryDBSafe(query: str, params: tuple = ()) -> List[Tuple]:
    """Executes a Parameterized SQL query on the database and returns the results.
    This function is designed to safely execute SQL queries with parameters, preventing SQL injection attacks.
    Inputs:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        query (str): The SQL query to be executed, with placeholders for parameters (e.g., "SELECT * FROM patients WHERE age > ?").
        params (tuple): A tuple of parameters to be safely substituted into the query (e.g., (30,)).
    Output:
        list: A list of tuples containing the results of the query.
    """
    results = cursor.execute(query, params).fetchall()  # Fetch all results from the executed query
    print(f"Query returned {len(results)} rows")  # Log the number of rows returned
    return results

### Define and load LLM Configuration and load it

In [ ]:
file_name = resourcePath + "config.json"  # Name of the configuration file
with open(file_name, "r") as file:  # Open the config file in read mode
    config = json.load(file)  # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")  # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Store API credentials in environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE  # Set API base URL as environment variable

print("Environment variables set successfully")  # Log successful environment variable setup
print(f"Using OpenAI API Key: {OPENAI_API_KEY[:4]}...")  # Log a portion of the API key for verification without exposing the full key
print(f"Using OpenAI API Base URL: {OPENAI_API_BASE}")  # Log the API base URL for verification

modelGenerator = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)  # Initialize the OpenAI chat model with specified parameters
modelValidator = ChatOpenAI(model="gpt-4o", temperature=0.0)  # Initialize a second OpenAI chat model for validation with zero temperature

print("Models initialized successfully")  # Log successful model initialization

## Tools Config

### Tools for the MCP Servers

#### `get_patient_profile`

In [ ]:
@tool("get_patient_profile")
def get_patient_profile(patient_id: str):
    """ "
    Retrieves information about a patient from the database based on their unique identifier.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose information is being requested.

    Output:
        dict: A dictionary containing the patient's information, including:
            - patient_id
            - first_name
            - last_name
            - birth_year
    """

    rows = queryDBSafe(
        """
            SELECT
            patient_id, first_name, last_name, birth_year, sex_at_birth,
            preferred_language, health_literacy_level, timezone, created_at
            FROM patients
            WHERE patient_id = ?
            """,
        (patient_id),
    )  # Execute a parameterized query to get patient info
    print(
        f"get_patient_profile: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(rows[0] if rows else {"error": f"Patient {patient_id} not found"})

#### `list_patient_encounters`

In [ ]:
@tool("list_patient_encounters")
def list_patient_encounters(patient_id: str, limit: int = 5):
    """ "
    Retrieves a list of encounters for a specific patient from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose encounters are being requested.

    Output:
        list: A list of dictionaries, each containing details about an encounter, including:
        - encounter_id
        - encounter_date
        - encounter_type
        - reason_for_visit
        - diagnosis_summary
        - provider_specialty
        - followup_instructions
        - care_team_contact
    """
    rows = queryDBSafe(
        """
            SELECT
               encounter_id, encounter_date, encounter_type, reason_for_visit,
               diagnosis_summary, provider_specialty, followup_instructions, care_team_contact
            FROM encounters
            WHERE patient_id = ?
            ORDER BY encounter_date DESC
            LIMIT ?
            """,
        (patient_id, max(1, min(int(limit), 10))),
    )  # Execute a parameterized query to get patient encounters
    print(
        f"list_patient_encounters: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(rows)

#### `get_recent_clinical_note`

In [ ]:
@tool("get_recent_clinical_note")
def get_recent_clinical_note(patient_id: str, note_type: str = "visit_note"):
    """ "
    Retrieves a list of recent clinical notes for a specific patient from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose clinical notes are being requested.

    Output:
        list: A list of dictionaries, each containing details about a clinical note, including:
        - note_id
        - encounter_id
        - note_date
        - author
        - note_content
    """
    rows = queryDBSafe(
        """
            SELECT
               note_id, encounter_id, patient_id, note_type, note_text, created_at, author_role
            FROM clinical_notes
            WHERE patient_id = ? AND note_type = ?
            ORDER BY created_at DESC
            LIMIT 1
            """,
        (patient_id, note_type),
    )  # Execute a parameterized query to get recent clinical notes
    print(
        f"get_recent_clinical_note: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(
        rows[0] if rows else {"error": f"No {note_type} found for patient {patient_id}"}
    )

#### `get_clinical_notes_for_encounter`

In [ ]:
def get_clinical_notes_for_encounter(encounter_id: str):
    """ "
    Retrieves all clinical notes associated with a specific encounter from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        encounter_id (str): The unique identifier of the encounter whose clinical notes are being requested.
    Output:
        list: A list of dictionaries, each containing details about a clinical note, including:
        - note_id
        - patient_id
        - note_type
        - note_content
        - created_at
        - author_role
    """
    rows = queryDBSafe(
        """
            SELECT
                note_id, encounter_id, patient_id, note_type, note_text, created_at, author_role
            FROM clinical_notes
                    WHERE encounter_id = ?
            ORDER BY created_at DESC
            """,
        (encounter_id,),
    )  # Execute a parameterized query to get clinical notes for an encounter
    print(
        f"get_clinical_notes_for_encounter: Retrieved {len(rows)} rows for encounter_id={encounter_id}"
    )  # Log the number of rows retrieved
    return toJson(rows)

#### `get_labs`

In [ ]:
def get_labs(
    patient_id: str,
    test_name: Optional[str] = None,
    limit: int = 5,
):
    """ "
    Retrieves all lab results associated with a specific patient from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose lab results are being requested.
    Output:
        list: A list of dictionaries, each containing details about a lab result, including:
        - lab_id
        - encounter_id
        - lab_name
        - lab_value
        - lab_units
        - reference_range
        - lab_date
    """

    query = (
        """
            SELECT
                lab_id, encounter_id, patient_id, lab_name, lab_value, lab_units, reference_range, lab_date
            FROM labs
                    WHERE patient_id = ?
            """
        + (f"AND test_name = ?" if test_name else "")
        + """
            ORDER BY result_date DESC
            LIMIT ?
            """
    )
    rows = queryDBSafe(
        query, (patient_id, limit)
    )  # Execute a parameterized query to get labs for a patient
    print(
        f"get_labs: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(rows)

#### `get_medications`

In [ ]:
@tool("get_medications")
def get_medications(patient_id: str, status: str = "active"):
    """
    Retrieves all medication records associated with a specific patient from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose medication records are being requested.
        status (str): Filter medications by status ("active", "stopped", or "all"). Default is "active".
    Output:
        list: A list of dictionaries, each containing details about a medication record, including:
        - medication_id
        - encounter_id
        - medication_name
        - dosage
        - frequency
        - route
        - start_date
        - end_date
    """

    if status not in ("active", "stopped", "all"):
        status = "active"

    rows = queryDBSafe(
        """
            SELECT
                 med_id, rxnorm_code, med_name, dose, route, frequency,
                start_date, end_date, status, indication, prescriber_specialty
            FROM medications
                    WHERE patient_id = ?
                    """
        + (f"AND status = ?" if status != "all" else "")
        + """
            ORDER BY start_date DESC
            """,
        (patient_id, status),
    )  # Execute a parameterized query to get medications for a patient
    print(
        f"get_medications: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(rows)

#### `get_allergies`

In [ ]:
@tool("get_allergies")
def get_allergies(patient_id: str):
    """ "
    Retrieves all allergy records associated with a specific patient from the database.
    Args:
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        patient_id (str): The unique identifier of the patient whose allergy records are being requested.
    Output:
        list: A list of dictionaries, each containing details about an allergy record, including:
        - allergy_id
        - allergen
        - reaction
        - severity
        - recorded_date
    """
    rows = queryDBSafe(
        """
            SELECT
                allergy_id, substance, reaction, severity, recorded_date
            FROM allergies
                    WHERE patient_id = ?
            ORDER BY recorded_date DESC
            """,
        (patient_id,),
    )  # Execute a parameterized query to get allergies for a patient
    print(
        f"get_allergies: Retrieved {len(rows)} rows for patient_id={patient_id}"
    )  # Log the number of rows retrieved
    return toJson(rows)

#### `lookup_lab_education`

In [ ]:
@tool("lookup_lab_education")
def lookup_lab_education(test_name: str) -> str:
    """Looks up patient-friendly explanations for lab tests from the reference CSV data.
    Inputs:
        lab_name (str): The name of the lab test to look up (e.g., "Hemoglobin A1c").
    Output:
        Optional[str]: A patient-friendly explanation of the lab test, or None if not found.
    """
    normalized_lab_name = normalizeText(
        test_name
    )  # Normalize the input lab name for consistent matching
    df = labExplain.copy()
    df["_k"] = df["test_name_normalized"].astype(str).map(normalizeText)

    hit = df[df["_k"] == normalized_lab_name]
    if hit.empty:
        hit = df[df["_k"].str.contains(normalized_lab_name, na=False)]

    if hit.empty:
        return toJson({"error": f"No lab education found for '{test_name}'"})

    row = hit.iloc[0].drop(labels=["_k"]).to_dict()
    return toJson(row)

#### `lookup_medication_education`

In [ ]:
@tool("lookup_medication_education")
def lookup_medication_education(med_name: str) -> str:
    """Looks up patient-friendly explanations for medications from the reference CSV data.
    Inputs:
        med_name (str): The name of the medication to look up (e.g., "Metformin").
    Output:
        Optional[str]: A patient-friendly explanation of the medication, or None if not found.
    """
    normalized_med_name = normalizeText(
        med_name
    )  # Normalize the input medication name for consistent matching
    df = medEdu.copy()
    df["_k"] = df["med_name_normalized"].astype(str).map(normalizeText)

    hit = df[df["_k"] == normalized_med_name]
    if hit.empty:
        hit = df[df["_k"].str.contains(normalized_med_name, na=False)]

    if hit.empty:
        return toJson({"error": f"No medication education found for '{med_name}'"})

    row = hit.iloc[0].drop(labels=["_k"]).to_dict()
    return toJson(row)

#### `lookup_trusted_source`

In [ ]:
@tool("lookup_trusted_source")
def lookup_trusted_source(source_id: str) -> str:
    """Looks up information about a trusted source from the reference CSV data.
    Inputs:
        source_id (str): Identifier for the trusted source (e.g., "501").
    Output:
        Optional[str]: A description of the trusted source, or None if not found.
    """
    hit = trustedSources[trustedSources["source_id"] == source_id]
    if hit.empty:
        return toJson({"error": f"source_id {source_id} not found"})
    return toJson(hit.iloc[0].to_dict())

#### `policyRoute`

In [ ]:
@tool("policyRoute")
def policyRoute(user_text: str) -> str:
    """
    Apply safety guardrails to a user query using `safety_policy_rules.csv` and return a routing decision.

    This tool implements a lightweight, deterministic policy layer that helps the agent decide whether to:
    - answer normally ("answer"),
    - refuse unsafe requests such as diagnosis/dose changes ("refuse"),
    - escalate urgent symptom descriptions ("escalate_emergency" or "escalate_clinician").

    Matching approach (MVP):
    - Normalizes the user text and checks whether each rule's `pattern_or_topic` appears as a substring.
    - If multiple rules match, selects the highest-priority action using:
      escalate_emergency > escalate_clinician > refuse > answer.

    Args:
        user_text (str): The raw user input/query to evaluate.

    Returns:
        str: A JSON string with keys:
            - action: "answer" | "refuse" | "escalate_emergency" | "escalate_clinician"
            - rule_id: matched rule identifier (if any)
            - template: standard response template for refusal/escalation (if any)
            - matched_rules: list of all matched rules with their actions/topics
        If no rules match, returns:
            {"action": "answer", "matched_rules": []}
    """
    normalized_text = normalizeText(user_text)
    rules = policyRules.to_dict(orient="records")

    matches = []
    for r in rules:
        topic = normalizeText(r.get("pattern_or_topic", ""))
        if topic and topic in normalized_text:
            matches.append(r)

    priority = {
        "escalate_emergency": 3,
        "escalate_clinician": 2,
        "refuse": 1,
        "answer": 0,
    }

    if not matches:
        return toJson({"action": "answer", "matched_rules": []})

    matches_sorted = sorted(
        matches,
        key=lambda r: priority.get(r.get("agent_action", "answer"), 0),
        reverse=True,
    )
    best = matches_sorted[0]

    out = {
        "action": best["agent_action"],
        "rule_id": best["rule_id"],
        "template": best["standard_response_template"],
        "matched_rules": [
            {
                "rule_id": m["rule_id"],
                "action": m["agent_action"],
                "topic": m["pattern_or_topic"],
            }
            for m in matches_sorted
        ],
    }
    return toJson(out)

## Agent Config

### Agent State

This TypedDict maintains all information needed throughout the agent's execution, including conversation history, policy decisions, and intermediate results.

In [ ]:
class AgentState(TypedDict):
    """
    Defines the state structure for the LangGraph agent workflow.

    This TypedDict maintains all information needed throughout the agent's execution,
    including conversation history, policy decisions, and intermediate results.
    """

    patient_id: str
    user_query: str

    decision: str
    policy_rule_id: Optional[str]
    policy_template: Optional[str]

    messages: List[BaseMessage]
    step: int
    max_steps: int

    citations: List[Dict[str, Any]]  # To track sources of information for transparency
    draft_answer: Optional[str]  # To hold the LLM-generated response before finalization
    final_answer: Optional[str]  # To hold the validated and polished response ready for user delivery
    error: List[str]  # To capture any errors encountered during processing for debugging and user feedback

In [ ]:
REACT_SYSTEM_PROMPT = """
You are an advanced clinical assistant AI specializing in EHR analysis. Your goal is to provide evidence-based, concise insights by analyzing patient records.
You must operate using the Thought-Action-Observation loop:
1.  **Thought:** Reason about the user's question, identifying necessary information (e.g., labs, vitals, patient history) and potential gaps in the current EHR data.
2.  **Action:** Use the provided tools to fetch required data using available tools:
    get_patient_profile,
    list_patient_encounters,
    get_recent_clinical_note,
    get_clinical_notes_for_encounter,
    get_labs,
    get_medications,
    get_allergies,
    lookup_lab_education,
    lookup_medication_education,
    lookup_trusted_source
3.  **Observation:** Analyze the output of the action. If data is insufficient, repeat the Thought-Action-Observation loop no more than 5 times to gather more information. Always consider safety implications and apply the `policy_route` tool to guide your responses.
4.  **Final Answer:** Provide a concise summary, diagnosis, or recommendation based on clinical guidelines (e.g., ADA, AHA). Always apply the `policy_route` tool to evaluate safety implications before taking any action.
"""

### `init_state`

Creates the initial state of the Agent for a single run of ReAct.

In [ ]:
def init_state(patient_id: str, user_query: str, max_steps: int = 5) -> AgentState:
    return AgentState({
        "patient_id": patient_id,
        "user_query": user_query,

        "decision": "answer",
        "policy_rule_id": None,
        "policy_template": None,
        "messages": [
            SystemMessage(content=REACT_SYSTEM_PROMPT),
            HumanMessage(
                content=f"patient_id={patient_id}\nUser question: {user_query}"
            ),
        ],
        "step": 0,
        "max_steps": max_steps,
        "citations": [],
        "draft_answer": None,
        "final_answer": None,
        "error": [],
    })

### Tool list

In [ ]:
TOOLS: List[BaseTool] = [
    get_patient_profile,
    list_patient_encounters,
    get_recent_clinical_note,
    get_clinical_notes_for_encounter,
    get_labs,
    get_medications,
    get_allergies,
    lookup_lab_education,
    lookup_medication_education,
    lookup_trusted_source
]

llmGeneratorTools = modelGenerator.bind_tools(TOOLS)  # Bind the defined tools to the model generator for use in the agent workflow

### `policyGateNode`

In [ ]:
def policyGateNode(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    Policy gate node that evaluates user queries against safety rules.

    Runs the policyRoute tool on the user's input message and stores the routing decision
    in the state for downstream nodes to respect.

    Args:
        state (Dict[str, Any]): The current agent state containing:
            - messages: List of chat messages

    Returns:
        Dict[str, Any]: Updated state with:
            - policy_decision: The parsed routing decision from policyRoute
            - messages: Original messages (unchanged)
    """
    try:
        route = json.loads(policyRoute.invoke({"user_text": state["user_query"]}))
        return {
            "decision": route.get("action", "answer"),
            "policy_rule_id": route.get("rule_id"),
            "policy_template": route.get("template"),
        }
    except Exception as e:
        return {"errors": state.get("errors", []) + [f"policy_gate failed: {e}"]}

In [ ]:
def agentNode(state: AgentState) -> Dict[str, Any]:
    """
    Main agent node that generates a response based on the conversation history and policy decision.

    This node uses the llmGeneratorTools to generate a response. It respects the policy decision
    made in the policyGateNode and can choose to refuse, escalate, or answer based on that decision.

    Args:
        state (AgentState): The current state of the agent, including:
            - messages: List of chat messages
    Returns:
        Dict[str, Any]: Updated state with:
            - draft_answer: The raw response generated by the LLM before validation
    """
    # Emergency short-circuit: finalize will enforce template; still produce a draft to stop loop
    if state.get("decision") == "escalate_emergency" and state.get("policy_template"):
        return {
            "draft_answer": state["policy_template"],
            "messages": state["messages"],  # still write something explicit
            "step": state["step"],
        }

    # Hard-stop
    if state["step"] >= state["max_steps"]:
        forced = (
            "I may not have enough information yet to answer confidently. "
            "I can explain what your records show or share general information. "
            "For medical advice (especially medication changes) or urgent concerns, please contact your clinician."
        )
        return {
            "draft_answer": forced,
            "messages": state["messages"],
            "step": state["step"],
        }

    # Invoke LLM with tools
    ai_msg = llmGeneratorTools.invoke(state["messages"])

    # Explicitly build updated messages + step (no in-place mutation reliance)
    new_messages = list(state["messages"]) + [ai_msg]
    new_step = state["step"] + 1

    # If no tool calls, treat as draft answer and stop looping
    if not getattr(ai_msg, "tool_calls", None):
        return {
            "messages": new_messages,
            "step": new_step,
            "draft_answer": ai_msg.content,
        }

    # Tool calls exist: continue loop, but MUST still return an update
    return {
        "messages": new_messages,
        "step": new_step,
    }

### `toolExecNode`

In [ ]:
PATIENT_TOOLNAMES = {
    "get_patient_profile",
    "list_patient_encounters",
    "get_recent_clinical_note",
    "get_clinical_notes_for_encounter",
    "get_labs",
    "get_medications",
    "get_allergies",
}

In [ ]:
def _cap_limit(args: Dict[str, Any], max_limit: int = 10) -> Dict[str, Any]:
    """
    Cap a tool call's 'limit' argument to control data volume and prompt size.

    Args:
        args (Dict[str, Any]): Tool argument dictionary (may include a 'limit' key).
        max_limit (int, optional): Maximum allowed limit value. Defaults to 10.

    Returns:
        Dict[str, Any]: A copied args dictionary with 'limit' coerced to int and capped to max_limit
        when present; otherwise returns args unchanged.
    """
    out = dict(args)  # copy args so we don't mutate the original dict
    if "limit" in out and out["limit"] is not None:
        try:
            out["limit"] = min(int(out["limit"]), max_limit)  # cap numeric limit
        except Exception:
            out["limit"] = max_limit  # fallback if limit isn't parseable
    return out

In [ ]:
def toolExecutionNode(state: AgentState) -> Dict[str, Any]:
    """
    Execute tool calls produced by the tool-enabled LLM and append results back into the message trace.

    This node:
    - reads tool calls from the last AI message,
    - enforces patient scoping by overwriting patient_id for sensitive tools,
    - caps limit parameters for controlled retrieval,
    - executes each tool from the approved tool map,
    - appends each tool output as a ToolMessage observation,
    - optionally extracts citations from lab/med education tool outputs.

    Inputs:
        state (AgentState): Current state containing at least:
            - patient_id: used to enforce patient scoping
            - messages: last message may contain tool_calls
            - citations/errors: existing lists to extend

    Outputs:
        Dict[str, Any]: State updates containing:
            - messages: updated message list including ToolMessages (tool observations)
            - citations: updated list with extracted citation metadata (if any)
            - errors: updated list with any tool execution / blocking errors
    """
    last = state["messages"][-1]  # last AI message should contain tool_calls
    tool_calls = getattr(last, "tool_calls", []) or []  # tool call specs from LLM
    tool_map = {t.name: t for t in TOOLS}  # whitelist: tool name -> tool callable

    new_messages = list(state["messages"])  # copy messages (avoid in-place mutation)
    new_citations = list(state.get("citations", []))  # copy citations
    new_errors = list(state.get("errors", []))  # copy errors

    for tc in tool_calls:
        name = tc.get("name")
        args = tc.get("args", {}) or {}
        tool_id = tc.get(
            "id"
        )  # required so ToolMessage can be associated to the correct call

        # Block any tool not in our allowlist (security)
        if name not in tool_map:
            new_errors.append(f"Blocked unknown tool call: {name}")
            new_messages.append(
                ToolMessage(
                    content=f"Blocked unknown tool: {name}",
                    name=name or "unknown",
                    tool_call_id=tool_id,
                )
            )
            continue

        # Critical: prevent cross-patient data access by enforcing patient_id
        if name in PATIENT_TOOLNAMES:
            args["patient_id"] = state["patient_id"]

        # Cap retrieval volume (limits prompt size and reduces accidental overexposure)
        if name in {"get_labs", "list_patient_encounters"}:
            args = _cap_limit(args, max_limit=10)

        try:
            # Execute tool (returns JSON string in our implementation)
            result_str = tool_map[name].invoke(args)
            if not isinstance(result_str, str):
                result_str = json.dumps(result_str, ensure_ascii=False, default=str)

            # Optional: extract citation metadata from education tools
            if name in {"lookup_lab_education", "lookup_medication_education"}:
                try:
                    data = json.loads(result_str)
                    if isinstance(data, dict) and "error" not in data:
                        cit = {
                            "source_id": data.get("source_id"),
                            "citation_url": data.get("citation_url"),
                            "used_for": (
                                "lab education"
                                if name == "lookup_lab_education"
                                else "med education"
                            ),
                        }
                        if cit.get("source_id") or cit.get("citation_url"):
                            new_citations.append(cit)
                except Exception:
                    pass  # if parsing fails, skip citation extraction silently (MVP-friendly)

            # Append tool observation for the agent to use in the next step (ReAct "Observation")
            new_messages.append(
                ToolMessage(content=result_str, name=name, tool_call_id=tool_id)
            )

        except Exception as e:
            err = f"{name} failed: {e}"
            new_errors.append(err)
            new_messages.append(
                ToolMessage(content=err, name=name, tool_call_id=tool_id)
            )

    return {
        "messages": new_messages,
        "citations": new_citations,
        "errors": new_errors,
    }  # explicit updates required by LangGraph

In [ ]:
def should_continue(state: AgentState) -> str:
    """
    Determines the next step in the agent workflow based on the current state.

    Inputs:
        state (AgentState): The current state of the agent, which includes:
        - draft_answer: If present, indicates the agent has produced an answer and can finalize.
        - messages: The conversation history, where the last message may contain tool_calls.

    Returns:
        str: The next node label for LangGraph routing:
          - "final" if a draft_answer exists (stop looping and finalize)
          - "tool" if the last AI message contains tool_calls (execute tools next)
          - "agent" otherwise (continue by calling the agent again)
    """
    if state.get("draft_answer"):
        return "final"  # agent has produced an answer; exit loop

    last = state["messages"][-1]  # inspect the most recent AI message
    if getattr(last, "tool_calls", None):
        return "tool"  # tools requested; execute them

    return "agent"  # no answer yet and no tools requested; ask agent again

In [ ]:
def finalPolicyOverrideNode(state: AgentState) -> Dict[str, Any]:
    """
    Applies the final policy override based on the decision made in the policy gate.

    Inputs:
        state (AgentState): Current state containing:
          - decision: policy decision (answer/refuse/escalate_*)
          - policy_template: standardized refusal/escalation message (if applicable)
          - draft_answer: agent-generated answer (if applicable)

    Outputs:
        Dict[str, Any]: State update with:
          - final_answer: the policy template if decision requires it; otherwise the draft_answer,
            or a conservative fallback if no draft exists.
    """
    decision = state.get("decision", "answer")
    template = state.get("policy_template")

    # If policy requires refusal/escalation, override any LLM output
    if decision in ("escalate_emergency", "refuse", "escalate_clinician") and template:
        return {"final_answer": template}

    # Otherwise return the model's draft answer (or a fallback)
    return {
        "final_answer": state.get("draft_answer")
        or "I’m not sure how to answer that yet."
    }

## Agent Workflow

In [ ]:
# Build the state graph for the agent workflow
graph = StateGraph(AgentState)

# Register nodes: policy gate -> ReAct agent -> tool execution -> final policy override
graph.add_node(
    "policy", policyGateNode
)  # evaluates user query against safety rules and decides routing
graph.add_node(
    "agent", agentNode
)  # generates responses and decides when to call tools based on messages and policy decision
graph.add_node(
    "tool", toolExecutionNode
)  # executes tools and appends observations, citations, errors to state
graph.add_node(
    "final", finalPolicyOverrideNode
)  # ensures final answer respects safety overrides

graph.set_entry_point(
    "policy"
)  # start the workflow at the policy gate to evaluate the user's query before any agent processing

# Define edges with conditional routing based on policy decisions and agent outputs
graph.add_conditional_edges(
    "policy",
    lambda s: "final" if s.get("decision") == "escalate_emergency" else "agent",
    {"final": "final", "agent": "agent"},
)

# After the agent node runs, decide whether to execute tools (if tool_calls exist) or finalize (if draft_answer exists), otherwise loop back to agent for another reasoning step
graph.add_conditional_edges(
    "agent", should_continue, {"tool": "tool", "agent": "agent", "final": "final"}
)

graph.add_edge(
    "tool", "agent"
)  # after executing tools, return to the agent to process the new observations and continue reasoning
graph.add_edge("final", END)  # after finalizing the answer, end the workflow
agent_app = (
    graph.compile()
)  # compile the graph into an executable application that can be called with an initial state to run the workflow

In [ ]:
# render and display the LangGraph workflow diagram in the notebook
display(
    Image(agent_app.get_graph().draw_mermaid_png())
)  # complete the code to define the LangGraph workflow to visualize

In [ ]:
pid = "P001"    # complete the code to define the patient ID
query = "What does my Hemoglobin A1C result mean?"    # complete the code to define the patient query

state = init_state(pid, query, max_steps=2)    # complete the code to define the number of iterations for the ReAct loop

print("State contents:", state)
print("Agent app type:", type(agent_app))

out = agent_app.invoke(state)
print(out["final_answer"])
